In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('../data/featured_dataset.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"Loaded: {df.shape}")
print(f"Features: {df.shape[1]}")

In [ ]:
df['target_request_rate'] = df['request_rate'].shift(-1)
df['target_latency_p95'] = df['latency_p95'].shift(-1)
print("Created primary targets: next-step request_rate and latency_p95")

In [ ]:
df['target_replica_count'] = df['replica_count'].shift(-1)
print("Created secondary target: next-step replica_count")

In [ ]:
print(f"\nBefore removing NaN: {len(df)}")
df_target = df.dropna()
print(f"After removing NaN: {len(df_target)}")
print(f"Dropped: {len(df) - len(df_target)} (last row)")

In [ ]:
print("\nTarget validation:")
print(f"Target null values: {df_target[['target_request_rate', 'target_latency_p95', 'target_replica_count']].isnull().sum().sum()}")
print(f"\nTarget statistics:")
print(df_target[['target_request_rate', 'target_latency_p95', 'target_replica_count']].describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].scatter(df_target['request_rate'], df_target['target_request_rate'], alpha=0.3)
axes[0,0].plot([df_target['request_rate'].min(), df_target['request_rate'].max()], 
               [df_target['request_rate'].min(), df_target['request_rate'].max()], 'r--')
axes[0,0].set_xlabel('Current Request Rate')
axes[0,0].set_ylabel('Target Request Rate')
axes[0,0].set_title('Request Rate: Current vs Next-Step')

axes[0,1].scatter(df_target['latency_p95'], df_target['target_latency_p95'], alpha=0.3, color='orange')
axes[0,1].plot([df_target['latency_p95'].min(), df_target['latency_p95'].max()], 
               [df_target['latency_p95'].min(), df_target['latency_p95'].max()], 'r--')
axes[0,1].set_xlabel('Current Latency P95')
axes[0,1].set_ylabel('Target Latency P95')
axes[0,1].set_title('Latency P95: Current vs Next-Step')

axes[1,0].scatter(df_target['replica_count'], df_target['target_replica_count'], alpha=0.3, color='green')
axes[1,0].plot([df_target['replica_count'].min(), df_target['replica_count'].max()], 
               [df_target['replica_count'].min(), df_target['replica_count'].max()], 'r--')
axes[1,0].set_xlabel('Current Replica Count')
axes[1,0].set_ylabel('Target Replica Count')
axes[1,0].set_title('Replica Count: Current vs Next-Step')

df_target[['request_rate', 'target_request_rate']].iloc[:100].plot(ax=axes[1,1])
axes[1,1].set_title('Request Rate Time Series (First 100 samples)')
axes[1,1].legend(['Current', 'Target (Next-Step)'])

plt.tight_layout()
plt.savefig('../results/img/prediction_targets.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
feature_cols = [col for col in df_target.columns if col not in ['target_request_rate', 'target_latency_p95', 'target_replica_count']]
target_cols = ['target_request_rate', 'target_latency_p95', 'target_replica_count']

print(f"\nFeature columns: {len(feature_cols)}")
print(f"Target columns: {len(target_cols)}")
print(f"\nFeatures (first 10): {feature_cols[:10]}")
print(f"Targets: {target_cols}")

In [ ]:
df_target.to_csv('../data/prediction_ready_dataset.csv', index=False)
print(f"Saved: prediction_ready_dataset.csv ({len(df_target)} rows, {df_target.shape[1]} columns)")

In [ ]:
target_metadata = {
    'input_rows': len(df),
    'output_rows': len(df_target),
    'rows_dropped': len(df) - len(df_target),
    'total_columns': df_target.shape[1],
    'feature_columns': len(feature_cols),
    'target_columns': len(target_cols),
    'primary_targets': [
        'target_request_rate',
        'target_latency_p95'
    ],
    'secondary_targets': [
        'target_replica_count'
    ],
    'prediction_horizon': 'next_step',
    'target_statistics': {
        'target_request_rate': {
            'mean': float(df_target['target_request_rate'].mean()),
            'std': float(df_target['target_request_rate'].std()),
            'min': float(df_target['target_request_rate'].min()),
            'max': float(df_target['target_request_rate'].max())
        },
        'target_latency_p95': {
            'mean': float(df_target['target_latency_p95'].mean()),
            'std': float(df_target['target_latency_p95'].std()),
            'min': float(df_target['target_latency_p95'].min()),
            'max': float(df_target['target_latency_p95'].max())
        },
        'target_replica_count': {
            'mean': float(df_target['target_replica_count'].mean()),
            'std': float(df_target['target_replica_count'].std()),
            'min': float(df_target['target_replica_count'].min()),
            'max': float(df_target['target_replica_count'].max())
        }
    }
}

with open('target_metadata.json', 'w') as f:
    json.dump(target_metadata, f, indent=2)

print("\n=== PREDICTION TARGET SUMMARY ===")
print(f"Primary targets: {', '.join(target_metadata['primary_targets'])}")
print(f"Secondary targets: {', '.join(target_metadata['secondary_targets'])}")
print(f"Prediction horizon: {target_metadata['prediction_horizon']}")
print(f"Dataset rows: {target_metadata['output_rows']}")
print(f"Features: {target_metadata['feature_columns']}")
print(f"Targets: {target_metadata['target_columns']}")
print("\n=== TASK 1.3 COMPLETE ===")